# Task 02：MDP 与动态规划——学习与实验入口

## 本 Notebook 的学习边界

Task 01 只有一次动作，本任务加入状态转移。核心 backup 为：

$$
V^\pi(s)=\sum_a\pi(a\mid s)\sum_{s',r}p(s',r\mid s,a)\left[r+\gamma(1-\mathbb{1}_{terminal})V^\pi(s')ight].
$$

先用 `GridWorld` 理解状态、边界和终止，再依次运行 policy evaluation、policy iteration 和 value iteration。所有算法从 `src/` 导入；Notebook 中的打印函数只负责帮助观察。


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.gridworld import GridWorld

env = GridWorld()

print("All states:")
print(env.get_states())

All states:
[(0, 0), (0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3), (2, 0), (2, 1), (2, 2), (2, 3), (3, 0), (3, 1), (3, 2), (3, 3)]


In [5]:
tests = [
    ((0, 0), GridWorld.RIGHT),  # 正常移动
    ((0, 0), GridWorld.UP),     # 撞地图边界
    ((1, 0), GridWorld.RIGHT),  # 撞障碍物
    ((3, 2), GridWorld.RIGHT),  # 到达 Goal
]

for state, action in tests:
    next_state = env.get_next_state(state, action)

    reward = env.get_reward(
        state,
        action,
        next_state,
    )

    print(
        f"{state} "
        f"--{GridWorld.ACTION_NAMES[action]}--> "
        f"{next_state}, reward={reward}"
    )

(0, 0) --RIGHT--> (0, 1), reward=-1.0
(0, 0) --UP--> (0, 0), reward=-1.0
(1, 0) --RIGHT--> (1, 0), reward=-1.0
(3, 2) --RIGHT--> (3, 3), reward=10.0


In [7]:
def print_grid(env):
    for row in range(env.height):
        cells = []

        for col in range(env.width):
            state = (row, col)

            if state == env.start_state:
                cell = "S"

            elif state == env.goal_state:
                cell = "G"

            elif state in env.obstacles:
                cell = "#"

            else:
                cell = "."

            cells.append(f"{cell:^7}")

        print("".join(cells))

print_grid(env)

   S      .      .      .   
   .      #      .      .   
   .      .      .      .   
   .      .      .      G   


In [10]:
from src.policy_evaluation import (
    create_uniform_random_policy,
    policy_evaluation,
)
random_policy = create_uniform_random_policy(env)

print(random_policy((2, 2)))

values_random = policy_evaluation(
    env,
    random_policy,
    gamma=0.9,
)

values_random

{0: 0.25, 1: 0.25, 2: 0.25, 3: 0.25}


{(0, 0): -8.805788069898524,
 (0, 1): -8.54040948866338,
 (0, 2): -7.626327557278598,
 (0, 3): -7.094270377681699,
 (1, 0): -8.540409488663379,
 (1, 2): -6.189340376880075,
 (1, 3): -5.270780690626143,
 (2, 0): -7.626327557278597,
 (2, 1): -6.189340376880074,
 (2, 2): -3.977289009586553,
 (2, 3): -0.4268584914136441,
 (3, 0): -7.094270377681697,
 (3, 1): -5.270780690626142,
 (3, 2): -0.4268584914136442,
 (3, 3): 0.0}

In [14]:
def print_values(env, values):
    for row in range(env.height):
        cells = []

        for col in range(env.width):
            state = (row, col)

            if state in env.obstacles:
                cell = "#####"

            elif env.is_terminal(state):
                cell = "  G  "

            else:
                cell = f"{values[state]:5.2f}"

            cells.append(f"{cell:^9}")

        print("".join(cells))

print_values(
    env,
    values_random,
)

  -8.81    -8.54    -7.63    -7.09  
  -8.54    #####    -6.19    -5.27  
  -7.63    -6.19    -3.98    -0.43  
  -7.09    -5.27    -0.43      G    


In [16]:
state = (2, 2)

action_probs = random_policy(state)

bellman_value = 0.0

for action, prob in action_probs.items():

    next_state = env.get_next_state(
        state,
        action,
    )

    reward = env.get_reward(
        state,
        action,
        next_state,
    )

    q_value = (
        reward
        + 0.9 * values_random[next_state]
    )

    print(
        GridWorld.ACTION_NAMES[action],
        "next_state =",
        next_state,
        "reward =",
        reward,
        "Q =",
        round(q_value, 4),
    )

    bellman_value += prob * q_value

print()
print(
    "Bellman result:",
    bellman_value,
)

print(
    "Stored V(s):",
    values_random[state],
)

UP next_state = (1, 2) reward = -1.0 Q = -6.5704
DOWN next_state = (3, 2) reward = -1.0 Q = -1.3842
LEFT next_state = (2, 1) reward = -1.0 Q = -6.5704
RIGHT next_state = (2, 3) reward = -1.0 Q = -1.3842

Bellman result: -3.9772894907321734
Stored V(s): -3.977289009586553


In [17]:
state = (2, 2)

for action in env.ACTIONS:

    next_state = env.get_next_state(
        state,
        action,
    )

    reward = env.get_reward(
        state,
        action,
        next_state,
    )

    q_value = (
        reward
        + 0.9 * values_random[next_state]
    )

    print(
        f"{GridWorld.ACTION_NAMES[action]:5s}: "
        f"Q({state}, action) = {q_value:.4f}"
    )

print()
print(
    f"V({state}) = "
    f"{values_random[state]:.4f}"
)

UP   : Q((2, 2), action) = -6.5704
DOWN : Q((2, 2), action) = -1.3842
LEFT : Q((2, 2), action) = -6.5704
RIGHT: Q((2, 2), action) = -1.3842

V((2, 2)) = -3.9773


In [18]:
q_values = []

for action in env.ACTIONS:
    next_state = env.get_next_state(
        state,
        action,
    )

    reward = env.get_reward(
        state,
        action,
        next_state,
    )

    q_values.append(
        reward
        + 0.9 * values_random[next_state]
    )

print(
    "Average Q:",
    sum(q_values) / 4,
)

print(
    "V(s):",
    values_random[state],
)

Average Q: -3.9772894907321734
V(s): -3.977289009586553


In [21]:
from src.policy_iteration import policy_iteration

policy_pi, values_pi = policy_iteration(
    env,
    gamma=0.9,
    theta=1e-6,
)

print_values(
    env,
    values_pi,
)

   1.81     3.12     4.58     6.20  
   3.12    #####     6.20     8.00  
   4.58     6.20     8.00    10.00  
   6.20     8.00    10.00      G    


In [22]:
ACTION_SYMBOLS = {
    GridWorld.UP: "↑",
    GridWorld.DOWN: "↓",
    GridWorld.LEFT: "←",
    GridWorld.RIGHT: "→",
}

def print_policy(env, policy):
    for row in range(env.height):
        cells = []

        for col in range(env.width):
            state = (row, col)

            if state in env.obstacles:
                cell = "#"

            elif env.is_terminal(state):
                cell = "G"

            else:
                action = policy[state]
                cell = ACTION_SYMBOLS[action]

            cells.append(f"{cell:^7}")

        print("".join(cells))

print_policy(
    env,
    policy_pi,
)

   ↓      →      ↓      ↓   
   ↓      #      ↓      ↓   
   ↓      ↓      ↓      ↓   
   →      →      →      G   


In [23]:
state = env.reset()

trajectory = [state]
rewards = []

while True:

    action = policy_pi[state]

    next_state, reward, done = env.step(action)

    trajectory.append(next_state)
    rewards.append(reward)

    state = next_state

    if done:
        break

print("Trajectory:")
print(trajectory)

print()

print("Rewards:")
print(rewards)

Trajectory:
[(0, 0), (1, 0), (2, 0), (3, 0), (3, 1), (3, 2), (3, 3)]

Rewards:
[-1.0, -1.0, -1.0, -1.0, -1.0, 10.0]


In [24]:
gamma = 0.9

G = 0.0
discount = 1.0

for reward in rewards:
    G += discount * reward
    discount *= gamma

print("Discounted return G0:")
print(G)

print(
    "V*(start):",
    values_pi[env.start_state],
)

Discounted return G0:
1.809800000000001
V*(start): 1.8098


In [25]:
print(
    "Random policy V(start):",
    values_random[env.start_state],
)

print(
    "Optimal policy V(start):",
    values_pi[env.start_state],
)

Random policy V(start): -8.805788069898524
Optimal policy V(start): 1.8098


In [27]:
from src.value_iteration import value_iteration

values_vi, policy_vi = value_iteration(
    env,
    gamma=0.9,
    theta=1e-6,
)

print("Value Iteration Values:")
print_values(
    env,
    values_vi,
)

print("Value Iteration Policy:")
print_policy(
    env,
    policy_vi,
)

Value Iteration Values:
   1.81     3.12     4.58     6.20  
   3.12    #####     6.20     8.00  
   4.58     6.20     8.00    10.00  
   6.20     8.00    10.00      G    
Value Iteration Policy:
   ↓      →      ↓      ↓   
   ↓      #      ↓      ↓   
   ↓      ↓      ↓      ↓   
   →      →      →      G   


In [28]:
max_difference = max(
    abs(values_pi[state] - values_vi[state])
    for state in env.get_states()
)

print(
    "Max value difference:",
    max_difference,
)

Max value difference: 0.0


## 实验结论与提交要求

运行完本 Notebook 后，不要只保留图。请在对应 `notes/` 中记录随机种子、环境版本、关键超参数、最终指标、曲线文件和一个失败现象。结论必须区分“代码运行成功”和“算法表现更好”：前者由自检确认，后者需要多 seed 或控制变量实验支持。

提交前从仓库根目录运行：

```bash
python eval/run.py
```
